# M3 Keycloak JWT demo

Notebook didattico per mostrare il flow completo della nuova API Ordini:

1. Keycloak emette un access token con `client_credentials`.
2. Il client usa il token per leggere la lista ordini.
3. Il client legge il dettaglio di un ordine.
4. Il client crea un nuovo ordine con testata e righe.
5. Il client legge le righe dell'ordine creato.

Questo notebook assume che l'esempio sia avviato localmente.

## Parametri di connessione

Se stai usando la configurazione standard del repository, i valori di default sono:

- Keycloak: `http://localhost:8080`
- API base: `http://localhost:8001`
- Realm: `m3-jwt`
- Client id: `m3-fastapi-client`
- Client secret: `m3-fastapi-secret`

In [ ]:
import json
import os

import requests

KEYCLOAK_BASE_URL = os.getenv('KEYCLOAK_BASE_URL', 'http://localhost:8080')
API_BASE_URL = os.getenv('API_BASE_URL', 'http://localhost:8001')
REALM = os.getenv('REALM', 'm3-jwt')
CLIENT_ID = os.getenv('CLIENT_ID', 'm3-fastapi-client')
CLIENT_SECRET = os.getenv('CLIENT_SECRET', 'm3-fastapi-secret')

TOKEN_URL = f'{KEYCLOAK_BASE_URL}/realms/{REALM}/protocol/openid-connect/token'
ORDERS_URL = f'{API_BASE_URL}/orders'

def bearer_headers(access_token: str) -> dict[str, str]:
    return {'Authorization': f'Bearer {access_token}'}

def pretty(data):
    print(json.dumps(data, indent=2, ensure_ascii=False))

TOKEN_URL, ORDERS_URL

In [ ]:
token_response = requests.post(
    TOKEN_URL,
    data={
        'grant_type': 'client_credentials',
        'client_id': CLIENT_ID,
        'client_secret': CLIENT_SECRET,
    },
    timeout=30,
)
token_response.raise_for_status()
token_data = token_response.json()

access_token = token_data['access_token']
print('Token type:', token_data.get('token_type'))
print('Access token prefix:', access_token[:30] + '...')

In [ ]:
orders_response = requests.get(
    ORDERS_URL,
    headers=bearer_headers(access_token),
    timeout=30,
)
orders_response.raise_for_status()
orders_data = orders_response.json()

pretty(orders_data)
first_order_id = orders_data[0]['id'] if orders_data else None
print('First order id:', first_order_id)

In [ ]:
if first_order_id is None:
    raise RuntimeError('No orders available to inspect')

order_detail_response = requests.get(
    f'{ORDERS_URL}/{first_order_id}',
    headers=bearer_headers(access_token),
    timeout=30,
)
order_detail_response.raise_for_status()
order_detail_data = order_detail_response.json()

pretty(order_detail_data)

In [ ]:
created_order_response = requests.post(
    ORDERS_URL,
    headers={
        **bearer_headers(access_token),
        'Content-Type': 'application/json',
    },
    json={
        'customer': 'Gamma Srl',
        'order_date': '2026-05-18',
        'status': 'new',
        'lines': [
            {
                'sku': 'MOUSE-001',
                'description': 'Wireless mouse',
                'quantity': 4,
                'unit_price': 18.5,
            },
            {
                'sku': 'PAD-002',
                'description': 'Mouse pad',
                'quantity': 4,
                'unit_price': 4.25,
            },
        ],
    },
    timeout=30,
)
created_order_response.raise_for_status()
created_order_data = created_order_response.json()
created_order_id = created_order_data['id']

pretty(created_order_data)
print('Created order id:', created_order_id)

In [ ]:
created_lines_response = requests.get(
    f'{ORDERS_URL}/{created_order_id}/lines',
    headers=bearer_headers(access_token),
    timeout=30,
)
created_lines_response.raise_for_status()
created_lines_data = created_lines_response.json()

pretty(created_lines_data)

## Da osservare

- il token viene ottenuto senza password utente, solo con client e segreto;
- l'API accetta richieste solo se il JWT e valido;
- il client legge prima la lista, poi un dettaglio, poi crea un ordine nuovo;
- il notebook riproduce lo stesso flusso del client Python e della collezione Postman.